In [1]:
import pandas as pd, numpy as np
from pathlib import Path
import os
# ROOT dinámico — funciona en local, CI y cualquier entorno
ROOT = Path.cwd()
while not (ROOT / "requirements.txt").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)

SILVER = Path("output/ambiental/02-silver")
GOLD   = Path("output/ambiental/03-gold")
GOLD.mkdir(parents=True, exist_ok=True)
print("✅ Rutas OK")

✅ Rutas OK


In [2]:
df = pd.read_parquet(SILVER / "clean_aemet_provincia_fecha.parquet")

# La col 'anio' parece tener solo 2026 (API live) — verificar con fecha real
df["fecha"] = pd.to_datetime(df["fecha"])
df["anyo_real"] = df["fecha"].dt.year

print(f"Shape: {df.shape}")
print(f"Años reales en fecha: {sorted(df['anyo_real'].unique())}")
print(df.head(3).to_string())

Shape: (82, 12)
Años reales en fecha: [np.int32(2026)]
  cod_provincia provincia      fecha  n_estaciones  ta_media  tamin_media  tamax_media  prec_total  hr_media  ta_std  anio  anyo_real
0            01     Álava 2026-04-14            28      9.57         9.53        10.57         0.0     69.74    1.97  2026       2026
1            01     Álava 2026-04-15            61      6.97         6.24         7.41         0.0     74.07    2.65  2026       2026
2            02  Albacete 2026-04-14            24     11.18        11.13        11.78         0.0     75.00    3.15  2026       2026


In [3]:
df_gold = (
    df.groupby(["provincia", "anyo_real"], as_index=False)
    .agg(
        ta_media_anual   = ("ta_media",    "mean"),
        tamin_media_anual= ("tamin_media", "mean"),
        tamax_media_anual= ("tamax_media", "mean"),
        prec_total_anual = ("prec_total",  "sum"),
        hr_media_anual   = ("hr_media",    "mean"),
        n_obs            = ("n_estaciones","sum"),
    )
    .rename(columns={"anyo_real": "anyo"})
)

# Redondear
for c in ["ta_media_anual", "tamin_media_anual", "tamax_media_anual",
          "prec_total_anual", "hr_media_anual"]:
    df_gold[c] = df_gold[c].round(2)

print(f"Shape: {df_gold.shape}")
print(f"Años: {sorted(df_gold['anyo'].unique())}")
print(df_gold.head(5).to_string())

Shape: (41, 8)
Años: [np.int32(2026)]
   provincia  anyo  ta_media_anual  tamin_media_anual  tamax_media_anual  prec_total_anual  hr_media_anual  n_obs
0   A Coruña  2026           14.02              13.86              14.34               6.9           76.12    989
1   Albacete  2026           10.70              10.40              11.15               0.0           72.49     75
2   Alicante  2026            7.66               7.20               8.48               0.0           74.98    115
3    Almería  2026           14.68              14.30              15.07               0.0           49.92     61
4  Andalucía  2026            9.46               9.16               9.85               1.0           91.80    139


In [4]:
df_gold.to_parquet(GOLD / "gold_ambiental_provincia_anio.parquet", index=False)
df_gold.to_csv(GOLD    / "gold_ambiental_provincia_anio.csv",     index=False)

print(f"✅ Guardado en {GOLD}")
print(f"   gold_ambiental_provincia_anio → {df_gold.shape}")
print(f"\nDtypes:\n{df_gold.dtypes}")

✅ Guardado en output/ambiental/03-gold
   gold_ambiental_provincia_anio → (41, 8)

Dtypes:
provincia                str
anyo                   int32
ta_media_anual       float64
tamin_media_anual    float64
tamax_media_anual    float64
prec_total_anual     float64
hr_media_anual       float64
n_obs                  int64
dtype: object
